# KOSIS 통계표 수치 받기 (표 ID 지정)

팀에서 공유받은 통계표 목록(`data/*.csv`, 2,364개)에서 고른 표만 KOSIS OpenAPI로 받아 `data/` 아래에 저장한다. T1-01 EDA용.

| 저장 위치 | 내용 | 커밋 |
| --- | --- | --- |
| `data/raw/kosis/<조사명>/<orgId>_<tblId>_<주기>.json` | 수치(KOSIS 응답 그대로) + 같은 이름의 `.meta.json` | 안 함 (.gitignore) |
| `data/cache/kosis/` | API 응답 캐시. 다시 실행하면 여기서 읽는다 | 안 함 |
| `data/raw/kosis/_download_log.jsonl` | 표별 결과(ok / empty / failed)와 셀 수 | 안 함 |

- 키는 레포 루트 `.env`의 `KOSIS_API_KEY`에서 읽는다(Colab은 보안 비밀). 키는 출력하지 않고, 캐시에도 들어가지 않는다.
- `TABLE_IDS`에 받을 표 ID를 넣고 위에서부터 차례로 실행한다.

In [ ]:
# Setup: move to the repo root, load the API key, set run options.
import os
from pathlib import Path

root = Path.cwd()
while not (root / "kosis_chat").is_dir() and root != root.parent:
    root = root.parent
assert (root / "kosis_chat").is_dir(), "open this notebook from inside the repo"
os.chdir(root)
import sys
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # so kosis_chat imports when the kernel started in notebooks/

try:
    from google.colab import userdata  # Colab secrets
    os.environ.setdefault("KOSIS_API_KEY", userdata.get("KOSIS_API_KEY") or "")
except Exception:
    from dotenv import load_dotenv
    load_dotenv(root / ".env", override=False)

API_KEY = os.environ.get("KOSIS_API_KEY", "")
print("repo:", root)
print("KOSIS_API_KEY set:", bool(API_KEY))  # never print the value

# Tables to download: KOSIS table IDs (통계표ID) from the shared catalog in data/*.csv.
TABLE_IDS = [  # T1-01 EDA targets (plans/T1-01.md)
    "DT_1B8000G",      # 인구동향조사: 월.분기.연간 인구동향 (B0 기준 표, 단위 명·건 혼재)
    "DT_1B26006_A01",  # 국내인구이동통계: 인구이동률 (단위 %)
    "DT_1B83A35",      # 인구동향조사: 시도/시군구/월별 혼인 (월 단위)
    "DT_1BPA001",      # 장래인구추계: 성 및 연령별 추계인구 (다단 분류, 연령 묶음 겹침)
    "DT_1B81A24",      # 인구동향조사: 다태아 여부별 출생 (단위 빈칸)
]
SURVEYS_TO_FETCH = None    # filter by survey name; None = every table in TABLE_IDS
MAX_TABLES = None          # stop after this many tables per run (None = no limit)
LATEST_PERIODS = 10        # None = every period; 10 = only the latest 10 periods per period type
SLEEP_SECONDS = 0.3        # pause between real API calls
RETRY_FAILED = False       # True = retry tables that failed in an earlier run
CELL_LIMIT = 40_000        # KOSIS returns at most 40,000 cells per request

DATA_DIR = root / "data"
RAW_DIR = DATA_DIR / "raw" / "kosis"
CACHE_DIR = DATA_DIR / "cache" / "kosis"
CATALOG_GLOB = "*.csv"     # shared catalog files directly under data/
TABLES_PATH = RAW_DIR / "_targets.jsonl"
LOG_PATH = RAW_DIR / "_download_log.jsonl"
for d in (RAW_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# HTTP helper: cached GET, tolerant JSON parsing, KOSIS error handling. The key never reaches disk.
import hashlib
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request

BASE = "https://kosis.kr/openapi"
STOP_ERRORS = {"10", "11", "40", "41", "42"}  # missing/expired key, call/row quota, user limit: stop the run


class KosisError(Exception):
    def __init__(self, code, message):
        super().__init__(f"KOSIS error {code}: {message}")
        self.code = str(code)


class KosisStop(KosisError):
    """Quota or account limit: stop now and resume later."""


def parse_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Older KOSIS responses use unquoted keys: {err:"30",errMsg:"..."}
        return json.loads(re.sub(r'([{,])\s*([A-Za-z_][A-Za-z0-9_]*)\s*:', r'\1"\2":', text))


def http_get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "kosis-ai-nlp-summary/0.1"})
    for attempt in range(4):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp:
                return resp.read().decode("utf-8")
        except (urllib.error.URLError, TimeoutError) as exc:
            if attempt == 3:
                raise
            wait = 2 ** (attempt + 1)
            print(f"  network error ({exc}); retry in {wait}s")
            time.sleep(wait)


def kosis_get(path, params, fetch=None):
    """GET BASE/path with params (apiKey added here). Returns parsed JSON; raises KosisError on API errors."""
    cache_key = path + "?" + urllib.parse.urlencode(sorted(params.items()))
    cache_file = CACHE_DIR / (hashlib.sha256(cache_key.encode()).hexdigest()[:24] + ".json")
    if cache_file.exists():
        data = json.loads(cache_file.read_text(encoding="utf-8"))
    else:
        url = f"{BASE}/{path}?" + urllib.parse.urlencode({**params, "apiKey": API_KEY})
        data = parse_json((fetch or http_get)(url))
        time.sleep(SLEEP_SECONDS)
        is_error = isinstance(data, dict) and "err" in data
        if not is_error or str(data.get("err")) == "30":  # cache data and "no data", not transient errors
            cache_file.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    if isinstance(data, dict) and "err" in data:
        code, msg = str(data.get("err")), data.get("errMsg", "")
        raise (KosisStop if code in STOP_ERRORS else KosisError)(code, msg)
    return data

## 1. 공유 목록에서 대상 표 찾기

In [ ]:
# Look up TABLE_IDS in the shared catalog CSVs and write the targets file.
import csv

catalog = {}
for path in sorted(DATA_DIR.glob(CATALOG_GLOB)):
    with path.open(encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            catalog[row["통계표ID"]] = {
                "survey_nm": row["통계조사명"],
                "org_id": row["기관ID"],
                "org_nm": row["작성기관"],
                "tbl_id": row["통계표ID"],
                "tbl_nm": row["통계표명"],
                "list_path": row["통계표경로"],
                "periods": row["수록기간"],
                "unit": row["단위"],
                "url": row["통계표URL"],
            }
print(f"catalog: {len(catalog)} tables from {len(list(DATA_DIR.glob(CATALOG_GLOB)))} files")

missing = [t for t in TABLE_IDS if t not in catalog]
assert not missing, f"not in the catalog: {missing}"
targets = [catalog[t] for t in TABLE_IDS]
with TABLES_PATH.open("w", encoding="utf-8") as f:
    for rec in targets:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
for rec in targets:
    print(f"  {rec['tbl_id']:<14} {rec['survey_nm']} / {rec['tbl_nm']} ({rec['periods']})")

## 2. 수치 받기 → `data/raw/kosis/`

표마다 메타(분류·항목·수록 시점)를 먼저 받고, 수치를 주기(연·월 등)별로 받는다.
한 번에 4만 셀을 넘으면 시점 → 분류(1단, 2단, …) → 항목 순으로 잘게 나눠 다시 받고 하나로 합친다.
키 오류(10·11)나 호출 한도 오류(40~42)가 나면 멈추고, 다음에 다시 실행하면 이어서 받는다.

In [ ]:
# Download values table by table.
import datetime as dt

# PRD meta may label periods differently from the prdSe request codes; unknown labels are reported, not guessed.
PRD_SE_CODE = {"Y": "Y", "A": "Y", "년": "Y", "M": "M", "월": "M", "Q": "Q", "분기": "Q", "H": "H", "반기": "H",
               "D": "D", "일": "D", "F": "F", "IR": "IR", "부정기": "IR", "S": "S"}
PERIOD_TYPE = {"Y": "Y", "M": "M", "Q": "Q", "H": "H"}  # periods the B0 loader accepts


def table_meta(org_id, tbl_id, kind):
    return kosis_get("statisticsData.do", {"method": "getMeta", "type": kind, "orgId": org_id, "tblId": tbl_id,
                                           "format": "json", "jsonVD": "Y"})


def dims_and_items(org_id, tbl_id):
    """Classification levels in order ([(obj_id, [codes])]) and item ids, from the ITM meta."""
    rows = table_meta(org_id, tbl_id, "ITM")
    levels, items = {}, []
    for r in rows:
        if r.get("OBJ_ID") == "ITEM":
            items.append(r["ITM_ID"])
        else:
            sn = int(r.get("OBJ_ID_SN") or 0)
            levels.setdefault((sn, r["OBJ_ID"]), []).append(r["ITM_ID"])
    return [(obj, codes) for (_, obj), codes in sorted(levels.items())], items


def periods_by_type(org_id, tbl_id):
    """{'Y': ['2015', '2016', ...], 'M': [...]} from the PRD meta."""
    out = {}
    for r in table_meta(org_id, tbl_id, "PRD"):
        label = str(r["PRD_SE"]).strip()
        if label not in PRD_SE_CODE:
            raise KosisError("prd", f"unknown period label {label!r} in PRD meta; add it to PRD_SE_CODE")
        out.setdefault(PRD_SE_CODE[label], []).append(str(r["PRD_DE"]))
    return {k: sorted(set(v)) for k, v in out.items()}


def fetch_values(org_id, tbl_id, levels, prd_se, periods, fixed=None, items=None, all_items=()):
    """Fetch one period range; on 'over 40,000 cells' (err 31) split and merge the pieces.

    levels: [(obj_id, [codes])] in objL order; fixed: {level_no: code} already pinned by a split.
    Split order: periods -> classification levels (1, 2, ...) -> items.
    """
    fixed = fixed or {}
    params = {"method": "getList", "orgId": org_id, "tblId": tbl_id, "format": "json", "jsonVD": "Y",
              "prdSe": prd_se, "startPrdDe": periods[0], "endPrdDe": periods[-1],
              "itmId": " ".join(items) if items else "ALL"}
    for lv in range(1, len(levels) + 1):
        params[f"objL{lv}"] = fixed.get(lv, "ALL")
    try:
        return kosis_get("Param/statisticsParameterData.do", params)
    except KosisStop:
        raise
    except KosisError as exc:
        if exc.code == "30":
            return []
        if exc.code != "31":
            raise
    if len(periods) > 1:
        mid = len(periods) // 2
        return (fetch_values(org_id, tbl_id, levels, prd_se, periods[:mid], fixed, items, all_items)
                + fetch_values(org_id, tbl_id, levels, prd_se, periods[mid:], fixed, items, all_items))
    for lv, (_, codes) in enumerate(levels, 1):
        if lv not in fixed and len(codes) > 1:
            rows = []
            for code in codes:
                rows += fetch_values(org_id, tbl_id, levels, prd_se, periods, {**fixed, lv: code}, items, all_items)
            return rows
    if items is None and len(all_items) > 1:
        rows = []
        for item in all_items:
            rows += fetch_values(org_id, tbl_id, levels, prd_se, periods, fixed, [item], ())
        return rows
    raise KosisError("31", f"cannot split below {CELL_LIMIT} cells (period {periods[0]}, fixed {fixed})")


def safe_name(s):
    return re.sub(r'[\\/:*?"<>|\s]+', "_", s).strip("_")


def done_tables():
    done = set()
    if LOG_PATH.exists():
        for line in LOG_PATH.read_text(encoding="utf-8").splitlines():
            rec = json.loads(line)
            same_mode = rec.get("latest_periods") == LATEST_PERIODS  # a 5-period run does not count for a full run
            if (rec["status"] in ("ok", "empty") and same_mode) or (rec["status"] == "failed" and not RETRY_FAILED):
                done.add((rec["org_id"], rec["tbl_id"]))
    return done


def download_table(rec):
    org_id, tbl_id = rec["org_id"], rec["tbl_id"]
    levels, all_items = dims_and_items(org_id, tbl_id)
    out_dir = RAW_DIR / safe_name(rec["survey_nm"])
    out_dir.mkdir(parents=True, exist_ok=True)
    total, files = 0, []
    for prd_se, periods in periods_by_type(org_id, tbl_id).items():
        if LATEST_PERIODS:
            periods = periods[-LATEST_PERIODS:]
        rows = fetch_values(org_id, tbl_id, levels, prd_se, periods, all_items=all_items)
        if not rows:
            continue
        stem = f"{org_id}_{tbl_id}_{prd_se}"
        (out_dir / f"{stem}.json").write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
        meta = {
            "tbl_id": tbl_id,
            "tbl_nm": rows[0].get("TBL_NM") or rec["tbl_nm"],  # the loader checks it against the rows
            "survey_nm": rec["survey_nm"],
            "org_nm": rec["org_nm"],
            "period_type": PERIOD_TYPE.get(prd_se, prd_se),
            "source_url": rec["url"],
            "fetched_at": dt.date.today().isoformat(),
            "note": f"KOSIS OpenAPI; periods {periods[0]}..{periods[-1]}; list: {rec['list_path']}",
        }
        (out_dir / f"{stem}.meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
        total += len(rows)
        files.append(f"{out_dir.name}/{stem}.json")
    return total, files


def log(rec, status, cells=0, files=(), error=""):
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"org_id": rec["org_id"], "tbl_id": rec["tbl_id"], "survey_nm": rec["survey_nm"],
                            "status": status, "cells": cells, "latest_periods": LATEST_PERIODS, "files": list(files), "error": error,
                            "at": dt.datetime.now().isoformat(timespec="seconds")}, ensure_ascii=False) + "\n")


def run_download():
    catalog = [json.loads(l) for l in TABLES_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
    targets = [r for r in catalog if SURVEYS_TO_FETCH is None or r["survey_nm"] in SURVEYS_TO_FETCH]
    done = done_tables()
    todo = [r for r in targets if (r["org_id"], r["tbl_id"]) not in done]
    if MAX_TABLES:
        todo = todo[:MAX_TABLES]
    skipped = sum(1 for r in targets if (r["org_id"], r["tbl_id"]) in done)
    print(f"targets {len(targets)}, skipped (done, or failed earlier) {skipped}, this run {len(todo)}")
    stats = {"ok": 0, "empty": 0, "failed": 0}
    for i, rec in enumerate(todo, 1):
        try:
            cells, files = download_table(rec)
        except KosisStop as exc:
            print(f"stopped at {rec['tbl_id']}: {exc}. Run this cell again later to resume.")
            break
        except Exception as exc:  # keep going; the log keeps the reason
            log(rec, "failed", error=str(exc)[:300])
            stats["failed"] += 1
            print(f"[{i}/{len(todo)}] FAIL {rec['tbl_id']} {rec['tbl_nm'][:30]}: {str(exc)[:120]}")
            continue
        status = "ok" if cells else "empty"
        log(rec, status, cells, files)
        stats[status] += 1
        print(f"[{i}/{len(todo)}] {status:5} {rec['tbl_id']} {rec['tbl_nm'][:30]} ({cells:,} cells)")
    print(stats)
    return stats


run_download()

## 3. 받은 표가 챗봇 로더로 읽히는지 확인

In [ ]:
# Load a few downloaded tables with the B0 loader (format B: KOSIS JSON + meta.json).
from kosis_chat.data.loader import load_table
from kosis_chat.data.schema import SchemaError

samples = sorted(p for p in RAW_DIR.rglob("*.json") if not p.name.endswith(".meta.json"))[:5]
for p in samples:
    try:
        t = load_table(p)
        print(f"OK   {p.relative_to(root)}: {t.meta.tbl_nm} / {len(t.cells):,} cells")
    except SchemaError as exc:
        print(f"FAIL {p.relative_to(root)}: {exc.problems[:3]}")